# Quiz 04: Web APIs and JSON

This quiz covers web APIs and JSON, from lectures 18 and 19.

- You have 75 minutes. The quiz is open-book and open-notes
- Fork and clone this repository, and work in your clone
- The data files you need are already in the `data/` folder
- Run the cells from top to bottom, and run them all again before you submit
- Tasks 4, 6, 7, 8, 9 and 10 send real requests to live APIs
- Tasks 7 and 10 run `curl` in the terminal. Paste the commands and their output into the answer cells

Write your answers in the empty cells below each task. The bonus task is optional.


In [12]:
import json

import requests
import pandas as pd

## Task 1

Look at this URL:

```
https://api.worldbank.org/v2/country/URY/indicator/SP.POP.TOTL?format=json
```

Write the scheme, the host, the path and the query string, one on each line. Then write the full URL for France's population from 2015 to 2018.

**Your answer:**

Scheme: https

The Host: api.worldbank.org

The Path: v2/country/URY/indicator/SP.POP.TOTL

The Query String: format=json

Full URL: https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&date=2015:2018


## Task 2

Load `data/shuttle.json` with `json.load` into a variable called `reply`. Print:

1. the name of the stop
2. how many arrivals the reply lists
3. the wait for the second arrival, with its unit, for example `5 minutes`


In [13]:
# Task 2
with open("data/shuttle.json") as s:
    reply = json.load(s)

print(reply["stop"]["name"])
print(len(reply["arrivals"]))
print(f"{reply['arrivals'][1]['wait']} {reply['units']['wait']}")

Emory Point
3
11 minutes


## Task 3

Load `data/wb_electricity_ken.json` into a variable called `wb`. It holds the share of Kenya's (KEN) population with access to electricity. Like many APIs, the World Bank sends a list with two elements: metadata first, then the records. Print the number of records the metadata says there are, and check it against the length of the records list. Then print the value for 2020.


In [14]:
# Task 3
with open("data/wb_electricity_ken.json") as w:
    wb = json.load(w)

meta, records = wb

print("Metadata total:", meta["total"])
print("Records length:", len(records))
print("Count matches:", meta["total"] == len(records))

for record in records:
    if record["date"] == "2020":
        print("2020 value:", record["value"])

Metadata total: 13
Records length: 13
Count matches: True
2020 value: 71.5


## Task 4

Ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the current wind speed at Emory: latitude 33.79, longitude -84.32, and `wind_speed_10m` as the `current` variable in the query. Pass the three values as a dict to `params`, with `timeout=10`, and save the reply as `r`. Print `r.url` and `r.status_code`. Then print the wind speed with its unit.


In [15]:
# Task 4
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 33.79,
    "longitude": -84.32,
    "current": "wind_speed_10m"
}

r = requests.get(url, params=params, timeout=10)
print(r.url)
print(r.status_code)

data = r.json()
speed = data["current"]["wind_speed_10m"]
unit = data["current_units"]["wind_speed_10m"]
print(f"{speed} {unit}")

https://api.open-meteo.com/v1/forecast?latitude=33.79&longitude=-84.32&current=wind_speed_10m
200
8.2 km/h


## Task 5

Without running any code, write the full URL that this call sends. Some characters must be percent-encoded in a URL. Encode them as `requests` would (lecture 18).

```python
requests.get("https://api.example.com/v2/books",
             params={"author": "Toni Morrison", "year": "1970:1990", "format": "json"})
```


**Your answer:**

https://api.example.com/v2/books?author=Toni+Morrison&year=1970%3A1990&format=json


## Task 6

Make two bad requests and print the status code and the body (`r.json()`) of each:

1. The Task 4 request with latitude 9999
2. `https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json` (the indicator is misspelled)

For each, say in one sentence whether `raise_for_status()` would stop the script. Then write one line of code that would catch the second error.


In [16]:
# Task 6
url1 = "https://api.open-meteo.com/v1/forecast"
params1 = {
    "latitude": 9999,
    "longitude": -84.32,
    "current": "wind_speed_10m"
}

r1 = requests.get(url1, params=params1, timeout=10)
print(r1.status_code)
print(r1.json())

url2 = "https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json"
timeout2 = 10

r2 = requests.get(url2, timeout=timeout2)
print(r2.status_code)
print(r2.json())

400
{'error': True, 'reason': 'Latitude must be in range of -90 to 90°. Given: 9999.0.'}
200
[{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]


**Your answer:**

For the forecast request raise_for_status() would stop the script because the server returns a 400 Bad Request client error status code.

For the World Bank request raise_for_status() would not stop the script because the World Bank API returns a 200 OK status code with an error message inside the JSON body itself.

## Task 7

In the terminal, use `curl` to ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the weather in Montevideo: latitude -34.90, longitude -56.19, and `relative_humidity_2m` as the `current` variable. Paste into the answer cell, as text:

1. The `curl` command that prints only the status line, and its output
2. The `curl` command that saves the reply as `montevideo.json`, in the top folder of the repository
3. The Python command that prints the saved file indented, and the humidity with its unit


**Your answer:**
 
curl -sI "https://api.open-meteo.com/v1/forecast?latitude=-34.90&longitude=-56.19&current=relative_humidity_2m" | head -n 1
 
curl -s "https://api.open-meteo.com/v1/forecast?latitude=-34.90&longitude=-56.19&current=relative_humidity_2m" -o data/montevideo.json

with open("data/montevideo.json") as m:
    data = json.load(m)

print(json.dumps(data, indent=2))
print(f"{data['current']['relative_humidity_2m']}{data['current_units']['relative_humidity_2m']}")

## Task 8

The script `scripts/pull_data.py` downloads one World Bank indicator and saves two files in `data/raw/`. Change its four settings to collect the share of the population using the internet in Kenya, Ghana and Senegal, from 2015 to 2024, with `internet` as the output name. Find the indicator code at [data.worldbank.org/indicator](https://data.worldbank.org/indicator).

Run the script from the top folder of this repository, and paste the settings you used and the script's full output into the answer cell. Then answer briefly:

1. How many rows did you get, and why does the script save the raw reply as well as the CSV?
2. Look at Ghana's values in `data/raw/internet.csv`. Would you trust the value for 2019? Say why, and what you would do about it in an analysis.


**Your answer:**

Settings:

INDICATOR = "IT.NET.USER.ZS"

COUNTRIES = "KEN;GHA;SEN"

YEARS = "2015:2024"

OUTPUT_NAME = "internet"

Full Output:

Requesting https://api.worldbank.org/v2/country/KEN;GHA;SEN/indicator/IT.NET.USER.ZS
Saved 30 rows to data/raw/internet.csv
  country_code country  year      value
0          GHA   Ghana  2015  23.000000
1          GHA   Ghana  2016  28.000000
2          GHA   Ghana  2017  37.884201
3          GHA   Ghana  2018  43.000000
4          GHA   Ghana  2019  22.073200

Answers:

I got 5 rows and the script saves the raw reply as well as the CSV so that you can compare with the values you pulled and for reproducibility. No, I would not trust the value because it drastically differentiates from the pattern we observed where the value steadily increases from 2015-2018 before experiencing a weird drop-off in 2019 and then continuining to increase as if 2019 never happened. Something must be off here. 

## Task 9

[Frankfurter](https://frankfurter.dev/v1/) is a free API for exchange rates. Its latest rates are at `https://api.frankfurter.dev/v1/latest`, and it takes two options: `base`, the currency you convert from, and `symbols`, the currencies you want, separated by commas.

Using `requests` and `params`, ask for the latest rates from `USD` to `BRL` and `EUR`. Print `r.url`, the date of the rates, and the BRL rate.

For a range of dates, replace `latest` with `start..end`, for example `2020-01-06..2020-01-19` (two weeks, Monday to Sunday). Request that range with the same `params`. Print how many days of rates you got and the BRL rate on 15 January 2020. In one sentence, say why there are fewer than 14 days.


In [17]:
# Task 9
params2 = {"base": "USD", "symbols": "BRL,EUR"}

r = requests.get("https://api.frankfurter.dev/v1/latest", params=params2, timeout=10)
r.raise_for_status()
latest = r.json()

print(r.url)
print("Date:", latest["date"])
print("BRL rate:", latest["rates"]["BRL"])

r_range = requests.get("https://api.frankfurter.dev/v1/2020-01-06..2020-01-19", params=params, timeout=10)
r_range.raise_for_status()
history = r_range.json()

print("Days of rates:", len(history["rates"]))
print("BRL rate on 2020-01-15:", history["rates"]["2020-01-15"]["BRL"])

https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL%2CEUR
Date: 2026-11-19
BRL rate: 5.892
Days of rates: 10
BRL rate on 2020-01-15: 4.635


**Your answer:**
Central banks only release foreign exchange rates on business days, so the four weekend days across the two weeks are left out, leaving only 10 days.

## Task 10

Frankfurter's two options are `base` and `symbols` (Task 9). Run these three commands, paste each reply, and say in one sentence how the API treated the options in each. Then say whether checking the status code alone would have caught any of the problems.

```bash
curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
```


**Your answer:**

Replies:

{"amount":1.0,"base":"EUR","date":"2026-11-19","rates":{"BRL":6.4765}}%       

{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"AUD":1.49,"BRL":5.892,"CAD":1.43,"CHF":0.84261,"CNY":6.6667,"CZK":22.223,"DKK":6.7997,"EUR":0.90975,"GBP":0.76038,"HKD":7.8456,"HUF":336.09,"IDR":18261,"ILS":3.2887,"INR":93.3,"ISK":124.82,"JPY":159.97,"KRW":1328.45,"MXN":18.9183,"MYR":4.1125,"NOK":9.6866,"NZD":1.8095,"PHP":63.695,"PLN":4.0184,"RON":4.8647,"SEK":10.141,"SGD":1.2897,"THB":33.905,"TRY":50.468,"ZAR":17.2806}}%  

{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"BRL":5.892}}%

Answer: 
For the first command, the API silently ignored Base=USD and fell back to the default base currency, EUR, while applying symbols=BRL.

For the second command, because symbols is misspelled (spelled as symbol instead), the API ignored it and returned exchange rates for all available currencies instead of filtering to only BRL.

For the third command, the parameter values are case-insensitive, so the API automatically uppercased usd and brl and returned the intended USD to BRL rate.

All three requests returned a 200 OK HTTP status code because the API silently ignores unrecognized or misnamed query parameters and falls back to default values rather than raising an error. Therefore, checking the status code doesn't tell us or catch the problems that we are experiencing. 

## Bonus task

Write a `while` loop that reads `data/wb_gdppc_2023_page{n}.json`, starting at `n = 1`, and extends a list called `records`, which starts empty. Stop using the `pages` field from the metadata. Do not hard-code 3. Print `len(records)` at the end: it should match the `total` field, which is 265.

In [ ]:
# Bonus task

When you are done, run all the cells from the top one more time and save the notebook. Then stage, commit and push everything to your fork: `quiz04.ipynb`, `montevideo.json`, your edited `scripts/pull_data.py` and the two files in `data/raw/`. Post your fork's link on Canvas.

Best of luck!